# 2장 — 욕조의 수학: 저량과 유량 (핵심)

웹 챕터(2장)와 **같은 예제, 같은 수치**를 파이썬으로 계산합니다:
순유량 막대를 쌓아 저량 만들기, 면적과 끝값, 그리고 칸을 잘게 썰수록 진짜 면적(적분)에 가까워지는 모습.

> 🤖 **AI 튜터와 함께 공부하세요**: 아래 셀을 실행하면 화면 오른쪽 아래에 튜터 버튼이 뜹니다.
> 로컬 Ollama 가 꺼져 있어도 노트북 학습에는 지장이 없습니다.

In [ ]:
from tutor import chat
chat('ch2')  # 이 챕터를 아는 AI 튜터 (로컬 Ollama — 없어도 학습엔 지장 없음)

## 1. 막대를 쌓으면 저량이 된다

한 칸 = 한 달. 막대 높이 = 그달의 순유량(명/월), 막대 넓이 = 높이 × 1개월 = 그달 저량이 변한 양(명).

$$\text{저량}(t) = \text{저량}(0) + \int_0^t \big(\text{유입}(s) - \text{유출}(s)\big)\, ds$$

막대 그래프에서는 조각이 한 달이라 ∫ 대신 그냥 더하기로 쌓습니다. 처음 저량은 100명.

In [ ]:
S0 = 100
presets = {
    '일정한 유입': [20] * 12,
    '늘어나는 유입': [5 * (i + 1) for i in range(12)],
    '줄어드는 유입': [60 - 5 * i for i in range(12)],
    '오르내림': [40, 40, 20, 10, -10, -30, -40, -30, -10, 10, 30, 40],
}

def accumulate(net, s0=S0):
    # 웹 sim.ts accumulate 와 같은 계산: [시작, 1월 말, …, 12월 말]
    out = [s0]
    for v in net:
        out.append(out[-1] + v)
    return out

for name, net in presets.items():
    s = accumulate(net)
    print(f'{name:8s}  막대 면적 합 {sum(net):+5d}명  →  12월 말 {s[-1]}명   (월말: {s[1:]})')

# 웹 실험실(StockFlowLab)·본문 수치 자가검증
assert accumulate(presets['일정한 유입'])[-1] == 340
assert sum(presets['늘어나는 유입']) == sum(presets['줄어드는 유입']) == 390
assert accumulate(presets['늘어나는 유입'])[-1] == accumulate(presets['줄어드는 유입'])[-1] == 490
ud = accumulate(presets['오르내림'])
assert ud.index(max(ud)) == 4 and max(ud) == 210      # 4월 말 정점 210명
assert ud[-1] == 170
print('✅ 웹과 일치: 같은 면적(+390)이면 끝값(490)이 같다 · 오르내림의 정점은 4월 말 210명')

## 2. 거꾸로 읽기: 저량의 기울기 = 순유량

$$\frac{d\,\text{저량}}{dt} = \text{유입} - \text{유출}$$

월말 저량의 차이(한 달 동안의 기울기)를 구하면 막대가 그대로 돌아옵니다. 쌓기(적분)와 기울기 읽기(미분)가 서로를 되돌린다는 것,
미적분의 기본 정리의 직관이에요.

In [ ]:
for name, net in presets.items():
    s = accumulate(net)
    slopes = [s[k + 1] - s[k] for k in range(12)]   # 한 달 동안의 변화 = 기울기
    assert slopes == net
print('✅ 네 보기 모두: 저량의 월별 기울기 = 순유량 막대 (미분이 적분을 되돌린다)')

# StepQuiz: 처음 100명, 순유량 +20, −10, +30
assert accumulate([20, -10, 30])[1:] == [120, 110, 140]
print('계산 연습: 120 → 110 → 140명')

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(2, 4, figsize=(13, 5.2), sharex=True)
for col, (name, net) in enumerate(presets.items()):
    ax = axes[0][col]
    ax.bar(range(1, 13), net, color=['#10b981' if v >= 0 else '#f43f5e' for v in net])
    ax.axhline(0, color='#64748b', lw=1)
    ax.set_title(name)
    ax.set_ylim(-60, 60)
    s = accumulate(net)
    axes[1][col].plot(range(0, 13), s, marker='o', color='#059669', lw=2)
    axes[1][col].set_ylim(0, 600)
    axes[1][col].set_xlabel('월')
axes[0][0].set_ylabel('순유량(명/월)')
axes[1][0].set_ylabel('저량(명)')
fig.suptitle('위: 순유량 막대(높이 = 기울기)   아래: 쌓인 저량(면적 합 = 변화량)')
plt.tight_layout()
plt.show()

## 3. 더 잘게 썰수록 진짜 면적에 가까워진다

유입이 매끄럽게 줄어 t개월 시점에 $60 - 5t$ (명/월)이라면, 12개월 동안 쌓이는 진짜 양(곡선 아래 면적)은

$$\int_0^{12} (60 - 5t)\,dt = 60 \cdot 12 - 2.5 \cdot 12^2 = 360\text{명}$$

한 칸의 첫날 값으로 막대를 세우면 조금씩 많이 셉니다. 칸 폭을 반으로 줄일 때마다 오차가 어떻게 되는지 봅시다(5장 예고).

In [ ]:
def left_sum(width):
    n = round(12 / width)
    return sum((60 - 5 * (k * width)) * width for k in range(n))

exact = 60 * 12 - 2.5 * 12 ** 2
rows = []
for width in [1, 0.5, 0.25, 0.125, 1 / 16]:
    approx = left_sum(width)
    rows.append((width, approx, approx - exact))
    print(f'칸 폭 {width:>7.4f}개월  →  쌓인 양 {approx:8.3f}명   오차 {approx - exact:+7.3f}')

assert exact == 360
assert rows[0][1] == 390 and rows[1][1] == 375 and rows[2][1] == 367.5
for (w1, _, e1), (w2, _, e2) in zip(rows, rows[1:]):
    assert abs(e2 - e1 / 2) < 1e-9            # 칸 폭을 반으로 → 오차도 반으로
print('✅ 390 → 375 → 367.5 → … → 360: 칸 폭을 반으로 줄이면 오차도 반으로 (1차 정확도, 5장)')

## 4. 오늘의 발견

- **저량 = 처음 저량 + 순유량의 적분(면적)**. 막대 넓이의 단위: (명/월) × (월) = 명.
- **순유량 = 저량의 순간 변화율(기울기)**. 쌓기와 기울기 읽기는 서로를 되돌린다(미적분의 기본 정리).
- 같은 면적이면 끝값이 같다(늘어나는 유입·줄어드는 유입 모두 490명). 모양은 가는 길만 바꾼다.
- 정점은 순유량이 +에서 −로 바뀌는 곳(오르내림: 4월 말 210명).
- 칸을 잘게 썰수록 진짜 면적에 가까워지고, 칸 폭을 반으로 줄이면 오차도 반이 된다(5장 오일러법).

다음 장(3장 눈덩이)에서 유량이 저량을 보고 정해지는 첫 고리, 강화 루프를 봅니다.
**웹 챕터로 돌아가 연습 10문제를 풀면 진도가 채워져요.**